In [2]:
import numpy as np
import gymnasium as gym

import torch

import ray
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.core.rl_module.multi_rl_module import MultiRLModuleSpec
from ray.rllib.core.rl_module.rl_module import RLModuleSpec
from ray.rllib.core.rl_module import RLModule

from robobunting_gym import RoboBuntingEnv # the environment class

import os
import pprint

## Train Agent

In [3]:
# Hyperparameters for training
BATCH_SIZE = 256
GAMMA = 0.99
EPS_EXPLORATION = 0.2
TARGET_UPDATE = 10
NUM_EPISODES = 4000
TEST_INTERVAL = 25
LEARNING_RATE = 1e-5
RENDER_INTERVAL = 20


def env_creator(config=None):
    env = RoboBuntingEnv(
    )
    return env


temp_env = RoboBuntingEnv()

obs_space = temp_env.observation_space["p1"]
act_space = temp_env.action_space["p1"]

ray.shutdown()
ray.init()

config = (
    PPOConfig()
    .environment(env=RoboBuntingEnv, 
                 disable_env_checking=True)
    .env_runners(num_env_runners=5) # used to collect samples 
    .framework("torch")
    .training(
        lr = LEARNING_RATE,
        train_batch_size_per_learner=BATCH_SIZE,
        num_epochs=10,
    )
    # .evaluation(
    #     evaluation_interval=2, # every n interval
    #     evaluation_num_env_runners=2,
    #     evaluation_duration_unit="episodes",
    #     evaluation_duration=100,
    # )
    .multi_agent(
        policies={
                "p1",
                "p2"
            },
        policy_mapping_fn=(lambda agent_id, episode, **kwargs: agent_id),
        count_steps_by="env_steps",
    )
    .rl_module(
        rl_module_spec=MultiRLModuleSpec(
            rl_module_specs={
            "p1" : RLModuleSpec(
                    model_config={"fcnet_hiddens": [128, 128]},
            ),
            "p2" : RLModuleSpec(
                 model_config={"fcnet_hiddens": [128, 128]},
            ) 
        }),
    )
)

algo = config.build()


2026-04-10 16:09:15,061	INFO worker.py:2013 -- Started a local Ray instance.
/Users/bguta/gradschool/projects/roboBunting_study1/analysis/RoboBuntingGym/robobunting-gym/lib/python3.11/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2026-04-10 16:09:17,948	WARNING 1672181354.py:64 -- DeprecationWarning: `build` has been deprecated. Use `AlgorithmConfig.build_algo` instead. This will raise an error in the future!
2026-04-10 16:09:17,952	WARNING algorithm_config.py:5131 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migrati

In [4]:
for i in range(10):
    result = algo.train()
    #print(f"Iter {i+1}: reward = {result['evaluation']['env_runners']['episode_return_mean']}")

algo.stop()

In [5]:
# pprint.pp(result['env_runners'])

## Save the trained agent and then load it for testing

In [6]:
checkpoint_path = algo.save_to_path(os.getcwd() + "/raycheckpoint/")

In [7]:
env = RoboBuntingEnv() # Use your custom class

modules = RLModule.from_checkpoint(
    checkpoint_path + "/learner_group" + "/learner" + "/rl_module"
    
)

obs, infos = env.reset()
done = False
max_steps_test = 2000
steps = 0
total_reward = 0
while not done:
    steps+=1
    actions = {}
    
    for agent_id, agent_obs in obs.items():
        # Identify which module to use
        module_id = "p1" if agent_id == "p1" else "p2"
        module = modules[module_id]
        
        # RLModules expect a batch. We wrap the single obs in a list/tensor.
        # Use 'forward_inference' for deterministic testing (no exploration)
        obs_batch = torch.tensor([agent_obs], dtype=torch.float32) 
        
        # Run the forward pass
        action_dist_params = module.forward_inference({"obs": obs_batch}, explortaion=False)['action_dist_inputs'].numpy()[0]

        # We have continuous actions, we sample the normal distribution. 
        # We could also use the mean (greedy action) by just taking action_dist_params[0:1] without sampling.
        # action_dist_params --> 0=mean, 1=log(stddev)
        sampled_action = np.clip(
            np.random.normal(action_dist_params[0:1], np.exp(action_dist_params[1:2])),
            a_min=env.action_space[agent_id].low[0],
            a_max=env.action_space[agent_id].high[0],
        )

        actions[agent_id] = sampled_action

    # Step the environment
    obs, rewards, terminations, truncations, infos = env.step(actions)
    total_reward += rewards['p1']
    if (steps) % 100 == 0:
        env.render(bar_len=50)
    
    done = steps > max_steps_test

print(f"Total Reward: {total_reward/max_steps_test}")

/var/folders/mg/srrfzfkd6d7_fcq4wk7gy0lm0000gs/T/ipykernel_69656/2973931093.py:24: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  obs_batch = torch.tensor([agent_obs], dtype=torch.float32)


.......................P2...................P1......
.......................P2....................P1.....
.......................P2....................P1.....
.......................P2....................P1.....
.......................P2....................P1.....
.......................P2....................P1.....
.......................P2....................P1.....
.......................P2.....................P1....
.......................P2.....................P1....
.......................P2.....................P1....
.......................P2.....................P1....
.......................P2.....................P1....
.......................P2......................P1...
.......................P2......................P1...
.......................P2......................P1...
.......................P2......................P1...
.......................P2......................P1...
.......................P2......................P1...
.......................P2.....................